## NLP CW - Michelle Lo, Hetty Symes, Evelyn Nutton

RoBERTa base model

In [1]:
!pip install nlpaug
!pip install sacremoses


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 49.2 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/nlp-pcl-cw-main

Mounted at /content/drive
/content/drive/MyDrive/nlp-pcl-cw-main


In [3]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import transformers
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import pipeline, RobertaModel, AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, get_scheduler
from torch.optim import AdamW
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE, RandomOverSampler
import nlpaug.augmenter.word as naw
import sacremoses
import nltk
import math
from dataset.dont_patronize_me import DontPatronizeMe

nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [4]:
from torch import cuda
device = 'cuda' if cuda.is_available() else 'cpu'
print(device)

cuda


In [5]:
def set_seed(seed=42):
    import torch
    import random
    import numpy as np

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [6]:

train_df = pd.read_csv("train_dev_data/train_set.csv")
dev_df = pd.read_csv("train_dev_data/dev_set.csv")
dontpatroniseme = DontPatronizeMe(None, "test_data/task4_test.tsv")
dontpatroniseme.load_test()
official_test_df =  dontpatroniseme.test_set_df
train_df_split, val_df = train_test_split(
    train_df,
    test_size=0.15,
    stratify=train_df['label'],
    random_state=42
)
print(train_df_split.shape)
print(val_df.shape)
val_df.to_csv("train_dev_data/val_set.csv")



(7118, 7)
(1257, 7)


### Data Augmentation via Back Translation

In [ ]:

# # # Initialize backtranslation
# # back_aug = naw.BackTranslationAug(
# #     from_model_name='Helsinki-NLP/opus-mt-en-de',
# #     to_model_name='Helsinki-NLP/opus-mt-de-en',
# #     device='cuda',
# #     max_length=256
# # )

# # # Minority class only
# # underrep = train_df[train_df['label'] == 1].copy()
# # underrep = underrep.dropna(subset=['text'])

# # texts_list = underrep["text"].tolist()

# # augmented_texts = []

# # batch_size_aug = 32

# # # Backtranslation in batches
# # for i in range(0, len(texts_list), batch_size_aug):
# #     batch = texts_list[i:i + batch_size_aug]

# #     aug_batch = back_aug.augment(batch)

# #     # Ensure output is list
# #     if isinstance(aug_batch, str):
# #         aug_batch = [aug_batch]

# #     augmented_texts.extend(aug_batch)

# # # Clean augmentation outputs
# # augmented_texts = [
# #     str(t) for t in augmented_texts
# #     if t is not None and len(str(t)) > 0
# # ]

# # # Align dataframe length safely
# # min_len = min(len(underrep), len(augmented_texts))

# # augmented_df = underrep.iloc[:min_len].copy()
# # augmented_df["text"] = augmented_texts[:min_len]

# # # Merge dataset
# # train_df_augment = pd.concat([train_df, augmented_df], ignore_index=True)

# # # Show class distribution
# # print(train_df_augment['label'].value_counts())

# # train_df = train_df_augment




# # Initialize backtranslation
# back_aug = naw.BackTranslationAug(
#     from_model_name='Helsinki-NLP/opus-mt-en-de',
#     to_model_name='Helsinki-NLP/opus-mt-de-en',
#     device='cuda',
#     max_length=256
# )

# # Minority class only (training split only!)
# underrep = train_df_split[train_df_split['label'] == 1].copy()
# underrep = underrep.dropna(subset=['text'])

# texts_list = underrep["text"].tolist()

# augmented_texts = []

# batch_size_aug = 32

# # Backtranslation augmentation
# for i in range(0, len(texts_list), batch_size_aug):
#     batch = texts_list[i:i + batch_size_aug]

#     aug_batch = back_aug.augment(batch)

#     if isinstance(aug_batch, str):
#         aug_batch = [aug_batch]

#     augmented_texts.extend(aug_batch)

# # Clean outputs
# augmented_texts = [
#     str(t).strip()
#     for t in augmented_texts
#     if t is not None and len(str(t).strip()) > 0
# ]

# # Safe alignment
# min_len = min(len(underrep), len(augmented_texts))

# augmented_df = underrep.iloc[:min_len].copy()
# augmented_df["text"] = augmented_texts[:min_len]

# # Merge augmentation ONLY into training split
# train_df_augmented = pd.concat(
#     [train_df_split, augmented_df],
#     ignore_index=True
# )

# print(train_df_augmented['label'].value_counts())

# # Update training dataframe
# train_df_split = train_df_augmented

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/298M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/298M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/797k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/768k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

label
0    6443
1    1350
Name: count, dtype: int64


In [ ]:

print(len(underrep))
print(len(augmented_texts))

675
675


### Over Sampling

In [ ]:
# print(underrep.head())
# print(augmented_df.tail())
# train_df_split.to_csv("train_dev_data/train_split_set_aug.csv")

# train_df_split = pd.read_csv("train_dev_data/train_split_set_aug.csv")

# print(train_df_split['label'].value_counts())

# X_train = train_df_split[['text']]  # Feature columns
# y_train = train_df_split['label']  # Target column

# # Initialize the random oversampler
# ros = RandomOverSampler(random_state=42)

# # Apply oversampling
# X_resampled, y_resampled = ros.fit_resample(X_train, y_train)


      par_id      art_id   keyword country  \
1294    1326  @@24624935  homeless      ke   
7385    7538  @@25249109   refugee      za   
3733    3806  @@24824562  homeless      sg   
5858    5974   @@3988877   in-need      lk   
7635    7790   @@1872744   in-need      gh   

                                                   text  label  orig_label  
1294  """ We 've seen in the past that Kenyans who a...      1           4  
7385  "LONDON - Angelia Jolie has urged people to ""...      1           3  
3733  """ I accept his apology and I appreciate the ...      1           3  
5858  "The Interact Club is a service Oriented Organ...      1           4  
7635  "As a child , I have always been told to give ...      1           4  
      par_id      art_id        keyword country  \
4935    5031  @@26291433     vulnerable      pk   
8289    9400  @@23232293       disabled      gh   
5735    5848   @@1894012       homeless      ng   
8228    8495   @@4567045  poor-families      pk   
8339  

### Combining Resampled and Augmented Data into New Data Frame

In [7]:
# # Update the dataset with the resampled values
# train_df_split = pd.DataFrame(X_resampled, columns=X_train.columns)
# train_df_split['label'] = y_resampled

# train_df_split.to_csv("train_dev_data/train_split_set_aug_resampled.csv")


# Load pre-sampled, pre-augmented dataset
train_df_split = pd.read_csv("train_dev_data/train_split_set_aug_resampled.csv")
val_df = pd.read_csv("train_dev_data/val_set.csv")

# Verify the oversampling result
print(train_df_split['label'].value_counts())
train_df_split.head(10)


label
0    6443
1    6443
Name: count, dtype: int64


,Unnamed: 0,text,label
0,0,"Contrary to such measures , in Sri Lanka , the...",0
1,1,"""He said the city has become unaffordable for ...",0
2,2,"""On the same page as the excellent letter from...",0
3,3,Despite the fight by the national and county g...,0
4,4,""""""" Because it was easier and cheaper than Eur...",0
5,5,Hamelin needs another Pied Piper ! Rats return...,0
6,6,"@Bruisers ... Fair enough , India has had prob...",0
7,7,Masemola said the students were all from poor ...,0
8,8,The phenomenon of gaslighting the disabled can...,0
9,9,Further data shows 42% of Grande Prairie 's ho...,0


### Loading the Roberta Base Model

In [8]:
# Load the pre-trained model
checkpoint = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(checkpoint, truncation=True, do_lower_case=True)
pretrained_model = RobertaModel.from_pretrained(checkpoint, num_labels=2)
pretrained_model = pretrained_model.to(device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### PCLData class

In [9]:
class PCLData(Dataset):
    def __init__(self, data, tokenizer, max_len, test=False):
        self.tokenizer = tokenizer
        self.data = data.reset_index(drop=True)  # Important
        self.test = test
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        text = str(row["text"])
        text = " ".join(text.split())

        inputs = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        target = -1 if self.test else row["label"]

        return {
            'ids': inputs['input_ids'].squeeze(0),
            'mask': inputs['attention_mask'].squeeze(0),
            'targets': torch.tensor(target, dtype=torch.long)
        }

In [10]:
MAX_LEN = 256

In [11]:

print("TRAIN Dataset: {}".format(train_df_split.shape))
print("VAL Dataset: {}".format(val_df.shape))
print("DEV(dev) Dataset: {}".format(dev_df.shape))
train_split_dataset = PCLData(train_df_split, tokenizer, MAX_LEN)
val_dataset = PCLData(val_df, tokenizer, MAX_LEN)
dev_dataset = PCLData(dev_df, tokenizer, MAX_LEN)

dev_params = {'batch_size': 8, 'shuffle': False, 'num_workers': 0}
dev_loader = DataLoader(dev_dataset, **dev_params)

val_params = {
    'batch_size': 8,
    'shuffle': False,
    'num_workers': 0
}

validation_loader = DataLoader(val_dataset, **val_params)

TRAIN Dataset: (12886, 3)
VAL Dataset: (1257, 8)
DEV(dev) Dataset: (2094, 7)


### Fine Tuning Model

In [12]:

class RobertaClass(torch.nn.Module):
    def __init__(self):
        super(RobertaClass, self).__init__()
        self.l1 = pretrained_model
        self.pre_classifier = torch.nn.Linear(768, 768)
        self.dropout = torch.nn.Dropout(0.3)
        self.classifier = torch.nn.Linear(768, 2)

    def forward(self, input_ids, attention_mask):
        output_1 = self.l1(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        hidden_state = output_1.last_hidden_state
        pooler = hidden_state[:, 0]   # CLS token

        pooler = self.pre_classifier(pooler)
        pooler = torch.nn.ReLU()(pooler)
        pooler = self.dropout(pooler)

        output = self.classifier(pooler)
        return output

In [13]:
model = RobertaClass()
model.to(device)

RobertaClass(
  (l1): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((

In [14]:

# Creating the loss function and optimizer

loss_function = torch.nn.CrossEntropyLoss()


def calcuate_accuracy(preds, targets):
    n_correct = (preds==targets).sum().item()
    return n_correct

In [15]:

def train(model, epoch, optimizer, training_loader, scheduler=None):
    tr_loss = 0
    n_correct = 0
    steps = 0
    seen = 0

    model.train()

    for i, data in tqdm(enumerate(training_loader, 0)):

        ids = data['ids'].to(device, dtype=torch.long)
        mask = data['mask'].to(device, dtype=torch.long)
        targets = data['targets'].to(device, dtype=torch.long)

        preds = model(ids, mask)

        loss = loss_function(preds, targets)

        tr_loss += loss.item()

        _, pred_labels = torch.max(preds, dim=1)
        # probs = torch.softmax(preds, dim=1)[:,1]
        # pred_labels = (probs > 0.35).long()


        n_correct += calcuate_accuracy(pred_labels, targets)

        steps += 1
        seen += targets.size(0)


        if i % 5000 == 0 and steps > 0:
            curr_loss = tr_loss / steps
            curr_acc = (n_correct * 100) / seen
            print(f"Training Loss per 5000 steps: {curr_loss}")
            print(f"Training Accuracy per 5000 steps: {curr_acc}")

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # if scheduler is not None:
        #     scheduler.step()

    print(f'Total Accuracy for Epoch {epoch}: {(n_correct*100)/seen}')
    print(f"Training Loss Epoch: {tr_loss/steps}")
    print(f"Training Accuracy Epoch: {(n_correct*100)/seen}")

    return

In [16]:

def valid(model, validation_loader):
    model.eval()

    n_correct = 0
    seen = 0
    #steps = 0

    preds_model = torch.tensor([]).to(device)
    targets_model = torch.tensor([]).to(device)

    with torch.no_grad():
        for _, data in tqdm(enumerate(validation_loader, 0)):

            ids = data['ids'].to(device, dtype=torch.long)
            mask = data['mask'].to(device, dtype=torch.long)
            targets = data['targets'].to(device, dtype=torch.long)

            preds = model(ids, mask)

            _, pred_labels = torch.max(preds, dim=1)
            # probs = torch.softmax(preds, dim=1)[:,1]
            # pred_labels = (probs > 0.35).long()

            n_correct += calcuate_accuracy(pred_labels, targets)
            #steps+=1

            seen += targets.size(0)

            preds_model = torch.cat((preds_model, pred_labels))
            targets_model = torch.cat((targets_model, targets))

    epoch_accu = (n_correct * 100) / seen

    return epoch_accu, preds_model, targets_model

# Hyperparameter tuning

In [17]:

def train_with_hyperparameters(save_model_name, learning_rate, batch_size, epochs,
                               use_scheduler=False, gamma=0.9):

    set_seed(42)
    torch.cuda.empty_cache()

    train_params = {
        'batch_size': batch_size,
        'shuffle': True,
        'num_workers': 0
    }

    training_loader = DataLoader(train_split_dataset, **train_params)


    model = RobertaClass().to(device)

    optimizer = torch.optim.AdamW(
        params=model.parameters(),
        lr=learning_rate
        #weight_decay=0.01
    )

    scheduler = None
    if use_scheduler:
        scheduler = torch.optim.lr_scheduler.ExponentialLR(
            optimizer,
            gamma=gamma
        )

    for epoch in range(epochs):
        train(model, epoch, optimizer, training_loader)
        if scheduler is not None:
          scheduler.step()


    torch.save(model.state_dict(), f"models/{save_model_name}.pt")

    acc, preds, targets = valid(model, validation_loader)

    print("Accuracy on validation data = %0.2f%%" % acc)
    print(classification_report(
        targets.cpu().numpy(),
        preds.cpu().numpy()
    ))
        # Confusion matrix
    cm = confusion_matrix(targets.cpu().numpy(), preds.cpu().numpy())
    print("Confusion Matrix:")
    print(cm)

    # Plot confusion matrix
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Class 0", "Class 1"], yticklabels=["Class 0", "Class 1"])
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix")
    plt.show()

In [ ]:

batch_sizes = [8, 16, 32]
learning_rates = [1e-5, 2e-5,3e-5]
gamma_rates = [0.9]

model_id = 0

for batch_size in batch_sizes:
    for lr in learning_rates:

        print(f"\nModel {model_id}: Batch size {batch_size}, LR {lr}, no scheduler")

        train_with_hyperparameters(
            model_id,
            lr,
            batch_size,
            5,
            use_scheduler=False
        )

        model_id += 1

        print(f"\nModel {model_id}: Batch size {batch_size}, LR {lr}, scheduler gamma 0.9")

        train_with_hyperparameters(
            model_id,
            lr,
            batch_size,
            5,
            use_scheduler=True,
            gamma=0.9
        )

        model_id += 1

In [19]:
EPOCHS = 5
FINAL_LR = 2e-5
FINAL_TRAIN_BATCH_SIZE = 8
FINAL_GAMMA = 0.9

#1e-5, 8, scheduler f1 = 0.55
#1e-5, 8, no scheduler f1 = 0.53
#2e-5, 8, no scheduler f1 = 0.53
#2e-5, 8, scheduler f1 = 0.58
#2e-5, 16, scheduler f1 = 0.54
#2e-5, 16, scheduler f1 = 0.51
#3e-5, 8, scheduler f1 = 0.51


#train_with_hyperparameters("final_model", FINAL_LR, FINAL_TRAIN_BATCH_SIZE, EPOCHS, use_scheduler=True, gamma = FINAL_GAMMA)

In [20]:
full_train_df = pd.concat([train_df_split, val_df]).reset_index(drop=True)

print("Merged training size:", full_train_df.shape)
print(full_train_df['label'].value_counts())
train_dataset_full = PCLData(full_train_df, tokenizer, MAX_LEN)

train_loader_full = DataLoader(
    train_dataset_full,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

model = RobertaClass().to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

loss_function = torch.nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.ExponentialLR(
            optimizer,
            gamma=FINAL_GAMMA
        )

set_seed(42)
for epoch in range(EPOCHS):
    train(model, epoch, optimizer, train_loader_full)
    scheduler.step()



Merged training size: (14143, 8)
label
0    7581
1    6562
Name: count, dtype: int64


0it [00:00, ?it/s]

Training Loss per 5000 steps: 0.6843301057815552
Training Accuracy per 5000 steps: 75.0


1768it [02:30, 11.71it/s]


Total Accuracy for Epoch 0: 89.04758537792547
Training Loss Epoch: 0.2577710626844414
Training Accuracy Epoch: 89.04758537792547


...

Total Accuracy for Epoch 3: 99.44141978363855
Training Loss Epoch: 0.018783306152521394
Training Accuracy Epoch: 99.44141978363855


2it [00:00, 12.83it/s]

Training Loss per 5000 steps: 0.000544295588042587
Training Accuracy per 5000 steps: 100.0


1768it [02:27, 11.97it/s]

Total Accuracy for Epoch 4: 99.53333804709044
Training Loss Epoch: 0.014724740423490274
Training Accuracy Epoch: 99.53333804709044


In [21]:

torch.save({
      'model_state_dict': model.state_dict(),
      'optimizer_state_dict': optimizer.state_dict()
  }, "models/final_roberta_model.pt")

In [17]:
# Recreate model architecture
model = RobertaClass().to(device)

# Load checkpoint
checkpoint = torch.load("models/final_roberta_model.pt", map_location=device)

# Load weights into model
model.load_state_dict(checkpoint['model_state_dict'])

<All keys matched successfully>

In [19]:

def save_and_evaluate_dev(model, loader, path):
    model.eval()

    preds_list = []
    targets_list = []

    with torch.no_grad():
        for data in loader:

            ids = data['ids'].to(device)
            mask = data['mask'].to(device)
            targets = data['targets'].to(device)

            preds = model(ids, mask)

            # # ⭐ Threshold tuning (very important for PCL)
            # probs = torch.softmax(outputs, dim=1)[:, 1]
            # preds = (probs > 0.35).long()
            _, preds = torch.max(preds, dim=1)

            preds_list.extend(preds.cpu().numpy())
            targets_list.extend(targets.cpu().numpy())

    with open(path, mode='wt', encoding='utf-8') as f:
        f.write('\n'.join([str(p) for p in preds_list]))

    print(f"Predictions saved → {path}")

    # ✅ Evaluation
    print("\nDev Evaluation Report:")
    print(classification_report(targets_list, preds_list))

    # print("Macro F1:",
    #       f1_score(targets_list, preds_list, average="macro"))



print("✅ Model saved successfully")
save_and_evaluate_dev(
    model,
    dev_loader,
    "Results/dev-check.txt"
)

✅ Model saved successfully
Predictions saved → Results/dev-check.txt

Dev Evaluation Report:
              precision    recall  f1-score   support

           0       0.96      0.93      0.95      1895
           1       0.50      0.64      0.56       199

    accuracy                           0.90      2094
   macro avg       0.73      0.79      0.75      2094
weighted avg       0.92      0.90      0.91      2094



# Evaluation

In [ ]:
# print("Classification Report:")
# print(classification_report(targets.cpu().numpy(), preds.cpu().numpy()))

# # Confusion matrix
# cm = confusion_matrix(targets.cpu().numpy(), preds.cpu().numpy())
# print("Confusion Matrix:")
# print(cm)

# # Plot confusion matrix
# sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Class 0", "Class 1"], yticklabels=["Class 0", "Class 1"])
# plt.xlabel("Predicted")
# plt.ylabel("Actual")
# plt.title("Confusion Matrix")
# plt.show()

In [24]:

print(official_test_df.head())

# def create_eval_txt(dataset_loader, path):
#   model.eval()
#   preds_model = []

#   with torch.no_grad():
#       for _, data in tqdm(enumerate(dataset_loader, 0)):
#           ids = data['ids'].to(device, dtype = torch.long)
#           mask = data['mask'].to(device, dtype = torch.long)
#           token_type_ids = data['token_type_ids'].to(device, dtype=torch.long)
#           preds = model(ids, mask, token_type_ids).squeeze()

#           _, pred_labels = torch.max(preds.data, dim=1)
#           preds_model.extend(pred_labels.tolist())

#   with open(path, mode='wt', encoding='utf-8') as f:
#       f.write('\n'.join([str(p) for p in preds_model]))

def create_eval_txt(dataset_loader, path):
    model.eval()
    preds_model = []

    with torch.no_grad():
        for _, data in tqdm(enumerate(dataset_loader, 0)):

            ids = data['ids'].to(device, dtype=torch.long)
            mask = data['mask'].to(device, dtype=torch.long)

            preds = model(ids, mask)

            _, pred_labels = torch.max(preds, dim=1)
            # probs = torch.softmax(preds, dim=1)[:,1]
            # pred_labels = (probs > 0.35).long()
            preds_model.extend(pred_labels.cpu().tolist())


    with open(path, mode='wt', encoding='utf-8') as f:
        f.write('\n'.join([str(p) for p in preds_model]))

base_path = "Results"

test_params = {'batch_size': 8, 'shuffle': False, 'num_workers': 0}
official_test_data = PCLData(official_test_df, tokenizer, MAX_LEN, True)
official_test_loader = DataLoader(official_test_data, **test_params)

create_eval_txt(official_test_loader, base_path+"/test.txt")

  par_id      art_id     keyword country  \
0    t_0   @@7258997  vulnerable      us   
1    t_1  @@16397324       women      pk   
2    t_2  @@16257812     migrant      ca   
3    t_3   @@3509652     migrant      gb   
4    t_4    @@477506  vulnerable      ca   

                                                text  
0  In the meantime , conservatives are working to...  
1  In most poor households with no education chil...  
2  The real question is not whether immigration i...  
3  In total , the country 's immigrant population...  
4  Members of the church , which is part of Ken C...  


479it [00:15, 31.44it/s]
